**In this model we built complete pipeline of using OneHot Encoding to convert the categorical data into encoded(vectors of numericals).**

In [5]:
import pandas as pd
import numpy as np

**step-1** **=> read the data**

In [6]:
df = pd.read_csv('cars.csv')   # load your dataset
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


**step-2**  **=> understand the data**

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   brand          8128 non-null   object
 1   km_driven      8128 non-null   int64 
 2   fuel           8128 non-null   object
 3   owner          8128 non-null   object
 4   selling_price  8128 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 317.6+ KB


**step-3 => Trsain_test_split**

In [8]:
from sklearn.model_selection import train_test_split
X = df.drop(columns='selling_price')
y = df['selling_price']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

**step-4 => Handle Rare Brands**

In [10]:
counts = df['brand'].value_counts()
threshold = 100
rare_brands = counts[counts < threshold].index
X_train['brand'] = X_train['brand'].replace(rare_brands,'uncommon')
X_test['brand'] = X_test['brand'].replace(rare_brands,'uncommon')


**step-5 => OneHot Encoder**

In [14]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop='first',             # avoid dummy variable trap
    sparse_output=False,      # get normal array
    handle_unknown='ignore'   # avoid errors for unseen values
)

**step-6 => Encode Categorical Columns**

In [15]:
cat_cols = ['fuel', 'owner', 'brand']

X_train_cat = ohe.fit_transform(X_train[cat_cols])   # learn + transform
X_test_cat = ohe.transform(X_test[cat_cols])         # only transform

**step-7 => Convert Encoded Data to DataFrame**

In [16]:
feature_names = ohe.get_feature_names_out(cat_cols)

X_train_cat_df = pd.DataFrame(X_train_cat, columns=feature_names)
X_test_cat_df = pd.DataFrame(X_test_cat, columns=feature_names)

**step-8 => Remove Original Categorical Columns**

In [17]:
X_train_num = X_train.drop(columns=cat_cols)
X_test_num = X_test.drop(columns=cat_cols)

**step-9 => Combine Numerical + Encoded Data**

In [18]:
X_train_final = pd.concat([X_train_num.reset_index(drop=True), X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_num.reset_index(drop=True), X_test_cat_df], axis=1)

**step-9 => Train Model**

In [19]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_final, y_train)

LinearRegression()

**step-10 => Make Predictions**

In [20]:
y_pred = model.predict(X_test_final)

**step-11 => Evaluate Model**

In [21]:
from sklearn.metrics import r2_score

print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.543295461604036
